In [9]:
from huggingface_hub import login

login()

In [ ]:
import pandas as pd
from datasets import load_dataset
import zstandard as zstd
import requests
import io
import json

TRAIN_ROWS_NEEDED = 5000 # True Members
TEST_ROWS_NEEDED = 5000   # True Non-Members
DOMAIN = "PubMed Abstracts"

# TRAIN DATA (MEMBERS)
# Method: Hugging Face Datasets via Parquet

print(f"--- Fetching {TRAIN_ROWS_NEEDED} Train Rows (Members) ---")

# 1. Load the streaming Parquet branch
train_dataset = load_dataset(
    "monology/pile-uncopyrighted", 
    split="train",
    revision="refs/convert/parquet",
    streaming=True
)

# 2. Filter and extract
def is_pubmed(example):
    return example["meta"]["pile_set_name"] == DOMAIN

pubmed_train_stream = train_dataset.filter(is_pubmed)
train_members_list = list(pubmed_train_stream.take(TRAIN_ROWS_NEEDED))

# 3. Save to Parquet
df_train = pd.DataFrame(train_members_list)
df_train.to_parquet("../dataset/members.parquet")
print(f"Saved {len(df_train)} rows to dataset/members.parquet\n")


# TEST DATA (NON-MEMBERS)
# Method: Direct zstandard streaming
print(f"--- Fetching {TEST_ROWS_NEEDED} Test Rows (Non-Members) ---")

def stream_test_set(max_rows, domain):
    url = "https://huggingface.co/datasets/monology/pile-test-val/resolve/main/test.jsonl.zst"
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(response.raw)
    text_stream = io.TextIOWrapper(stream_reader, encoding='utf-8')
    
    extracted_data = []
    
    for line in text_stream:
        row = json.loads(line)
        if row.get("meta", {}).get("pile_set_name") == domain:
            extracted_data.append(row)
            if len(extracted_data) >= max_rows:
                break
                
    return extracted_data

# 1. Stream and extract
test_non_members_list = stream_test_set(TEST_ROWS_NEEDED, DOMAIN)

# 2. Save to Parquet
df_test = pd.DataFrame(test_non_members_list)
df_test.to_parquet("../dataset/non_members.parquet")
print(f"Saved {len(df_test)} rows to dataset/non_members.parquet\n")


# VERIFICATION
print("--- Final Verification ---")
check_train = pd.read_parquet("../dataset/members.parquet")
check_test = pd.read_parquet("../dataset/non_members.parquet")

print(f"Members Dataset (Train): {check_train.shape[0]} rows")
print(f"Non-Members Dataset (Test): {check_test.shape[0]} rows")
print("Ready for your Membership Inference Attack!")

--- Fetching 5000 Train Rows (Members) ---
Saved 5000 rows to dataset/members.parquet

--- Fetching 5000 Test Rows (Non-Members) ---
Saved 5000 rows to dataset/non_members.parquet

--- Final Verification ---
Members Dataset (Train): 5000 rows
Non-Members Dataset (Test): 5000 rows
Ready for your Membership Inference Attack!


In [11]:
df = pd.read_parquet("../dataset/members.parquet")

In [15]:
df.head()

,text,meta
0,PCI Alternative Using Sustained Exercise (PAUS...,{'pile_set_name': 'PubMed Abstracts'}
1,TiO2 nanotubes for bone regeneration.\nNanostr...,{'pile_set_name': 'PubMed Abstracts'}
2,Standardised protocol for primate faecal analy...,{'pile_set_name': 'PubMed Abstracts'}
3,Examination of factors affecting gait properti...,{'pile_set_name': 'PubMed Abstracts'}
4,Formulation and application of a biosurfactant...,{'pile_set_name': 'PubMed Abstracts'}
